In [6]:
!pip install tabpfn -q

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tabpfn import TabPFNClassifier
import torch

In [7]:
import os
os.environ["TABPFN_TOKEN"] ="API_KEY"

In [8]:
train_df = pd.read_csv("/content/GSE98320.csv")
val_df   = pd.read_csv("/content/GSE129166.csv")

X_train = train_df.drop(columns=["sample_id", "diagnosis"])
y_train = train_df["diagnosis"]
X_val   = val_df.drop(columns=["sample_id", "diagnosis"])[X_train.columns]
y_val   = val_df["diagnosis"]

In [9]:
# Impute any missing gene values (median fit on training data only)
imputer = SimpleImputer(strategy="median")
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val   = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)

class_labels = sorted(y_train.unique())
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [10]:
# TabPFN has no hyperparameters to grid-search -- this IS the model config.
# If you hit a feature-count error below, your installed TabPFN version has
# a lower feature cap than 767; see the fallback note after this cell.
def make_model():
    return TabPFNClassifier(device=device, random_state=42)

def print_metrics(y_true, y_pred, label):
    print(f"\n=== TabPFN — {label} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    for cls, pi, ri, fi in zip(class_labels, p, r, f):
        print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
    print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")

In [11]:
# Nested CV -- no inner grid search, just fit + predict per outer fold
outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
X_train_r, y_train_r = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
oof_pred = np.empty(len(y_train_r), dtype=object)

for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_train_r, y_train_r)):
    model = make_model()
    model.fit(X_train_r.iloc[tr_idx], y_train_r.iloc[tr_idx])
    oof_pred[te_idx] = model.predict(X_train_r.iloc[te_idx])
    print(f"  fold {fold+1}/10 done")

print_metrics(y_train_r, oof_pred, "Cross-Validation Performance (GSE98320)")

tabpfn-v3-classifier-v3_default.ckpt: reconstructing file:   0%|          |  0.00B /  213MB            

tabpfn-v3-classifier-v3_default.ckpt: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/33.0 [00:00<?, ?B/s]

  fold 1/10 done
  fold 2/10 done
  fold 3/10 done
  fold 4/10 done
  fold 5/10 done
  fold 6/10 done
  fold 7/10 done
  fold 8/10 done
  fold 9/10 done
  fold 10/10 done

=== TabPFN — Cross-Validation Performance (GSE98320) ===
Accuracy: 0.9153
  ABMR   | precision=0.8987  recall=0.8436  f1=0.8703
  NR     | precision=0.9286  recall=0.9574  f1=0.9427
  TCMR   | precision=0.8442  recall=0.8025  f1=0.8228
  MACRO  | precision=0.8905  recall=0.8678  f1=0.8786


In [12]:
# Fit on full training set once, then evaluate on validation (TCMR excluded)
final_model = make_model()
final_model.fit(X_train, y_train)

val_mask = y_val != "TCMR"
X_val_no_tcmr = X_val[val_mask]
y_val_no_tcmr = y_val[val_mask]
val_class_labels = [c for c in class_labels if c != "TCMR"]

print(f"\nExcluded {(~val_mask).sum()} TCMR sample(s), evaluating on {len(y_val_no_tcmr)} remaining samples")


Excluded 2 TCMR sample(s), evaluating on 75 remaining samples


In [13]:
val_pred = final_model.predict(X_val_no_tcmr)

print(f"\n=== TabPFN — Independent Validation Performance (GSE129166, TCMR excluded) ===")
print(f"Accuracy: {accuracy_score(y_val_no_tcmr, val_pred):.4f}")
p, r, f, _ = precision_recall_fscore_support(y_val_no_tcmr, val_pred, labels=val_class_labels, zero_division=0)
for cls, pi, ri, fi in zip(val_class_labels, p, r, f):
    print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")


=== TabPFN — Independent Validation Performance (GSE129166, TCMR excluded) ===
Accuracy: 0.9733
  ABMR   | precision=0.8824  recall=1.0000  f1=0.9375
  NR     | precision=1.0000  recall=0.9667  f1=0.9831
  MACRO  | precision=0.9412  recall=0.9833  f1=0.9603


In [14]:
val_pred_all = final_model.predict(X_val)

print(f"\n=== TabPFN — Independent Validation Performance (GSE129166, TCMR included) ===")
print(f"Accuracy: {accuracy_score(y_val, val_pred_all):.4f}")
p, r, f, _ = precision_recall_fscore_support(y_val, val_pred_all, labels=class_labels, zero_division=0)
for cls, pi, ri, fi in zip(class_labels, p, r, f):
    print(f"  {cls:6s} | precision={pi:.4f}  recall={ri:.4f}  f1={fi:.4f}")
print(f"  MACRO  | precision={np.mean(p):.4f}  recall={np.mean(r):.4f}  f1={np.mean(f):.4f}")


=== TabPFN — Independent Validation Performance (GSE129166, TCMR included) ===
Accuracy: 0.9740
  ABMR   | precision=0.8824  recall=1.0000  f1=0.9375
  NR     | precision=1.0000  recall=0.9667  f1=0.9831
  TCMR   | precision=1.0000  recall=1.0000  f1=1.0000
  MACRO  | precision=0.9608  recall=0.9889  f1=0.9735
